<div style='background:#000000;border-radius:10px;padding:26px 30px;border-top:6px solid #3777FF;border-bottom:3px solid #0EE071;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'>
<p style='color:#0EE071;margin:0;letter-spacing:2px;font-size:12px'>NOVA IMS · ENTERPRISE DATA SCIENCE BOOTCAMP 2026 · PROJETO D</p>
<h1 style='color:#FFFFFF;margin:6px 0 4px 0'>Telco Customer Churn · <span style='color:#3777FF'>Retention Management</span></h1>
<p style='color:#FFFFFF;margin:0;font-size:15px'>Notebook 02 — Modelos base, rede neuronal e ensembles · avaliação contra o benchmark IBM · priorização por valor</p>
</div>

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>Índice — Estrutura do notebook</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Implementa os <b style='color:#3777FF'>próximos passos</b> definidos na secção 14 do notebook 01.
Pré-requisito: correr primeiro <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>01_preprocessing_eda.ipynb</code> ou <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>python scripts/run_pipeline.py</code> (gera <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>preprocessing_config.joblib</code>).</p>
<p>Documentação de apoio: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>docs/models.md</code> (como funciona cada modelo) e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>docs/modelling.md</code> (protocolo, métricas e decisões). O mesmo fluxo corre sem notebook com <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>python scripts/run_modelling.py</code>.</p>
<ol style='margin:4px 0'>
<li>Setup do ambiente</li><li>Carregamento dos artefactos</li><li>Os modelos em avaliação</li>
<li>Tuning dos modelos base</li><li>Comparação em validação cruzada e ensembles</li>
<li>Satisfaction Score — com e sem</li><li>City com TargetEncoder</li>
<li>Avaliação final no holdout</li><li>Interpretação</li>
<li>Modelo de priorização: clientes estabelecidos (tenure &gt; 3)</li><li>Priorização dos clientes estabelecidos: P(churn) × CLTV</li><li>Clientes novos: regra de onboarding</li>
<li>Exportação</li><li>Conclusões para o relatório</li></ol></div></div>

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>1 · Setup do ambiente</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Mesmo setup do notebook 01. Dois módulos do projeto fazem o trabalho desta fase:</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>modelling</code>: modelos, tuning, ensembles, métricas, avaliação e priorização</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>model_plots</code>: gráficos da modelação, com a paleta do projeto</li></ul>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>N_ITER</code> controla o nº de combinações testadas por modelo no <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>RandomizedSearchCV</code>. Com 20, a secção 4 demora alguns minutos; baixar para 5 para testar depressa.</p></div></div>

In [ ]:
# --- Bibliotecas ---
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

# --- Encontrar a pasta do projeto (a que tem a pasta "src") ---
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# --- Módulos do próprio projeto (pasta src/telco_churn) ---
from telco_churn import features, model_plots, modelling, plots, retention, scoring
from telco_churn.config import ARTIFACTS_DIR, DATA_PROCESSED, TAB_DIR, ensure_dirs
from telco_churn.style import apply_theme, style_table

# --- Opções ---
warnings.filterwarnings("ignore")         # esconder avisos
pd.set_option("display.max_columns", 60)  # mostrar mais colunas nas tabelas

SHOW = True    # True = mostrar os gráficos (são sempre guardados em ficheiro)
N_ITER = 20    # nº de combinações testadas por modelo (menos = mais rápido)

ensure_dirs()
apply_theme()
print("Pasta do projeto:", ROOT)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>2 · Carregamento dos artefactos</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Carrega <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>preprocessing_config.joblib</code> (listas de features e target) e os CSV de treino, teste e Joined.
Os CSV são lidos com <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>keep_default_na=False</code> para preservar o literal <i>None</i> em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Offer</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Internet Type</code>.</p>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>City</code> não foi exportada nos CSV de treino/teste (alta cardinalidade); é recuperada da base unificada por <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Customer ID</code> só para o teste da secção 7.</p></div></div>

In [ ]:
# Carregar os dados preparados no notebook 01
train, test, joined, cfg = modelling.load_processed()

NUM = cfg["features_num"]   # colunas numéricas
CAT = cfg["features_cat"]   # colunas categóricas
TARGET = cfg["target"]      # coluna que queremos prever (churn)
FEATURES = NUM + CAT        # todas as colunas usadas pelo modelo

# X = colunas de entrada, y = resposta (churn)
X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print("Treino:", len(train), "clientes | taxa de churn:", round(y_train.mean(), 3))
print("Teste: ", len(test), "clientes | taxa de churn:", round(y_test.mean(), 3))
print("Joined:", len(joined), "clientes (sem resposta, só para prever)")
print(len(NUM), "numéricas +", len(CAT), "categóricas =", len(FEATURES), "features")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>3 · Os modelos em avaliação</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Comparamos <b style='color:#3777FF'>sete modelos base</b>, uma <b style='color:#3777FF'>linha de base</b> e <b style='color:#0EE071'>dois ensembles</b>. Os modelos base cobrem as grandes famílias de classificadores:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>Linear</b>: Logistic Regression, a referência simples e interpretável</li><li><b style='color:#3777FF'>Árvore</b> e <b style='color:#3777FF'>distância</b>: Decision Tree e k-Nearest Neighbors, os modelos das aulas 6 e 7</li><li><b style='color:#3777FF'>Rede neuronal</b>: Neural Network (MLP), que generaliza a regressão logística com camadas escondidas</li><li><b style='color:#3777FF'>Ensembles de árvores</b>: Random Forest e Extra Trees (bagging) e Gradient Boosting (boosting)</li></ul>
<p>A tabela mostra como funciona cada um e que pré-processador usa: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>linear</code> (com escala, para modelos que somam variáveis ou medem distâncias) ou <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>trees</code> (sem escala). A explicação completa, com os hiperparâmetros, está em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>docs/models.md</code>.</p></div></div>

In [ ]:
# Família, pré-processador e funcionamento de cada modelo
display(style_table(modelling.MODEL_INFO))
print("Membros dos ensembles:", ", ".join(modelling.ENSEMBLE_MEMBERS))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>3.1 · A rede neuronal (MLP)</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Um <b style='color:#3777FF'>multilayer perceptron</b> recebe as variáveis escaladas, passa-as por uma ou duas <b style='color:#3777FF'>camadas escondidas</b> de neurónios com ativação não linear (ReLU ou tanh) e termina num neurónio com <b style='color:#3777FF'>sigmoide</b>, que devolve P(churn). Os pesos são aprendidos por backpropagation (otimizador Adam), minimizando o log loss.</p>
<p>A ligação à regressão: uma <b style='color:#0EE071'>regressão logística é uma rede neuronal sem camadas escondidas</b>, um único neurónio sigmoide. As camadas escondidas permitem à rede criar as suas próprias combinações das variáveis. Uma rede de <i>regressão</i> (saída contínua, perda quadrática) não serve aqui, porque o alvo é binário: usamos a mesma arquitetura com saída sigmoide e log loss.</p>
<p>O tuning escolhe o nº de camadas e neurónios, a ativação, a regularização L2 (<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>alpha</code>) e a taxa de aprendizagem. O <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>early_stopping</code> separa 15% do treino e pára quando a accuracy nessa validação interna deixa de melhorar.</p></div></div>

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>3.2 · As nove métricas</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Cada modelo é avaliado com as mesmas nove métricas do notebook das aulas. As quatro primeiras usam as probabilidades; as cinco últimas precisam de uma decisão sim/não e usam o limiar 0,5.
A <b style='color:#3777FF'>ROC AUC</b> é a métrica de seleção, porque a equipa de retenção trabalha com uma lista ordenada; o <b style='color:#3777FF'>Brier</b> é o critério secundário, porque P(churn) multiplica o CLTV na priorização.</p></div></div>

In [ ]:
display(style_table(modelling.metrics_glossary()))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>4 · Tuning dos modelos base</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Cada modelo é um <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Pipeline([("prep", ...), ("clf", ...)])</code>: o pré-processador é ajustado <b style='color:#0EE071'>dentro de cada fold</b>, por isso não há fuga de informação na validação cruzada.</p>
<p>Procura aleatória (<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>RandomizedSearchCV</code>) com <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>StratifiedKFold</code> (5 folds) <b style='color:#0EE071'>só no treino</b>, métrica ROC AUC, para os sete modelos base. O holdout não é usado até à secção 8. Os espaços de procura estão em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>modelling.BASE_MODELS</code>.</p></div></div>

In [ ]:
# Procurar os melhores hiperparâmetros de cada modelo (validação cruzada com 5 folds)
# A rede neuronal pode avisar que não convergiu em algumas combinações: é esperado no tuning.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
best_params, tuning = modelling.tune_base_models(X_train, y_train, NUM, CAT, n_iter=N_ITER)

tuning.to_csv(TAB_DIR / "10_tuning_modelos_base.csv")
display(style_table(tuning.sort_values("cv_auc", ascending=False), precision=4, highlight="cv_auc"))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>5 · Comparação em validação cruzada e ensembles</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Os modelos afinados são combinados de duas formas, com cinco membros de famílias diferentes (linear, rede neuronal, bagging e boosting):</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#0EE071'>Soft Voting</b>: média simples das probabilidades</li><li><b style='color:#0EE071'>Stacking</b>: um meta-modelo logístico aprende o peso de cada membro a partir de previsões <i>out-of-fold</i></li></ul>
<p>Todos os modelos, base e ensembles, são avaliados no <b style='color:#3777FF'>mesmo 5-fold</b> do treino com as nove métricas. Para cada modelo guardamos a média e o desvio-padrão em validação, a média no treino (para medir o sobreajuste) e as <b style='color:#3777FF'>previsões out-of-fold</b>: cada cliente recebe a probabilidade do modelo que não o viu. É com elas que se fazem os gráficos desta secção, sem tocar no holdout.</p>
<p>Esta é a célula mais lenta do notebook: o stacking treina os cinco membros dentro de cada fold.</p>
<p><b style='color:#3777FF'>Nota sobre o código:</b> <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>modelling.cv_evaluate</code> usa <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>joblib.Parallel</code> para treinar os 5 folds ao mesmo tempo, um por núcleo do processador. O resultado é igual ao de um ciclo <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>for</code> sobre os folds; só demora menos. Não usamos <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>cross_validate</code> do sklearn porque precisamos, de uma só vez, das métricas de treino e de validação e das previsões out-of-fold.</p></div></div>

In [ ]:
# Duas linhas de base (Dummy e regra de negócio) + 7 modelos afinados + 2 ensembles
models = modelling.build_models(NUM, CAT, best_params)

# Todos no mesmo 5-fold: nove métricas e previsões out-of-fold.
# O Churn Score IBM entra como referência e é avaliado exatamente nos mesmos folds.
ibm = {modelling.BENCHMARK: train["Churn Score"].to_numpy() / 100}
cv_table, oof = modelling.cv_evaluate(models, X_train, y_train, benchmarks=ibm)
cv_table.to_csv(TAB_DIR / "11_cv_modelos.csv")

cv_view = modelling.metrics_view(cv_table)            # médias em CV (modelos x métricas)
cv_train = modelling.metrics_view(cv_table, "train")  # médias no treino
cv_std = cv_table[[m + "_std" for m in modelling.METRICS]]
cv_std.columns = list(modelling.METRICS)
display(style_table(cv_view, precision=3, highlight="auc"))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.1 · Diversidade entre modelos</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Um ensemble só ganha se os membros errarem em clientes diferentes. A matriz mostra a correlação entre as probabilidades out-of-fold de cada par de modelos: valores perto de 1 indicam modelos que "pensam" igual; valores mais baixos indicam erros diferentes, que se compensam na média.</p></div></div>

In [ ]:
corr = modelling.prediction_correlation(oof)
corr.to_csv(TAB_DIR / "11c_correlacao_modelos.csv")
model_plots.plot_diversity(corr, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.2 · ROC AUC e seleção do modelo final</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>O ponto de partida é o modelo de <b style='color:#0EE071'>maior ROC AUC médio em CV</b>, escolhido antes de olhar para o holdout. A barra de erro é o desvio-padrão entre folds. As referências aparecem ao lado dos modelos: a barra preta é o Churn Score IBM (nos mesmos folds) e a cinzenta é a regra de negócio sem ML (1 ponto por fator de risco: Month-to-Month, tenure ≤ 12 meses, fibra ótica).</p>
<p>Modelos a menos de um desvio-padrão do vencedor estão, na prática, <b style='color:#3777FF'>empatados</b>: a diferença está dentro do ruído entre folds. <b style='color:#0EE071'>Regra de parcimónia:</b> entre o melhor e os empatados com ele, escolhe-se o mais simples, segundo a ordem fixa em <code>modelling.COMPLEXITY_ORDER</code> (Logistic Regression &lt; Decision Tree &lt; kNN &lt; Random Forest &lt; Extra Trees &lt; Gradient Boosting &lt; MLP &lt; Soft Voting &lt; Stacking). A tabela abaixo mostra o modelo de maior AUC, os empatados e o escolhido.</p></div></div>

In [ ]:
model_plots.plot_cv_comparison(cv_table, show=SHOW)

# Regra de parcimónia: o mais simples entre o melhor e os empatados (as referências não contam)
BEST, selecao, motivo = modelling.select_parsimonious(cv_table)
selecao.to_csv(TAB_DIR / "11d_selecao_modelo.csv")
display(style_table(selecao, precision=4))
empatados = list(selecao.index[selecao["empatado_com_melhor"] & ~selecao["escolhido"]])

print("Modelo escolhido:", BEST, "| AUC", round(cv_table.loc[BEST, "auc"], 4),
      "±", round(cv_table.loc[BEST, "auc_std"], 4))
print("Porquê:", motivo)
print("Referências em CV - AUC IBM:", round(cv_table.loc[modelling.BENCHMARK, "auc"], 3),
      "| regra de negócio:", round(cv_table.loc[modelling.RULE_BASELINE, "auc"], 3))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.3 · As nove métricas, modelo a modelo</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Um painel por métrica e uma barra por modelo (média em CV, com o desvio-padrão). O <b style='color:#3777FF'>losango</b> é o valor no treino. Ensembles e modelo final a <b style='color:#0EE071'>verde</b>, modelos individuais a <b style='color:#3777FF'>azul</b>, linha de base a cinzento.</p>
<p>Como ler: a linha de base tem accuracy alta e precision/recall a zero, porque nunca sinaliza ninguém. Nas métricas com limiar 0,5, a maioria dos modelos tem precision acima do recall: com 28% de churn, o limiar 0,5 é exigente (ver 5.7).</p></div></div>

In [ ]:
model_plots.plot_metrics_panel(cv_view, std=cv_std, train=cv_train, best=BEST, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.4 · Grelha modelo × métrica</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>O mesmo resumo numa grelha. A cor indica a <b style='color:#3777FF'>posição</b> do modelo em cada métrica (azul mais forte = melhor), já invertida no log loss e no Brier; o número é o valor real. Ajuda a ver se um modelo é bom em tudo ou só nalgumas métricas.</p></div></div>

In [ ]:
model_plots.plot_metrics_heatmap(cv_view, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.5 · Sobreajuste: treino vs validação</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>ROC AUC no treino e em validação cruzada. Quanto maior a distância (Δ), mais o modelo decorou o treino. As florestas e o boosting têm sempre algum Δ, porque cada árvore aprende bem o seu treino; o que importa é o valor em CV, que é o que se espera em clientes novos.</p></div></div>

In [ ]:
model_plots.plot_overfitting(cv_table.drop(index=modelling.REFERENCES), show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.6 · Curvas ROC e Precision-Recall</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Curvas de todos os modelos com as previsões out-of-fold, mais o Churn Score IBM a tracejado preto. O modelo final está a <b style='color:#0EE071'>verde e mais grosso</b>.</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>ROC</b>: % de churners apanhados (vertical) contra % de clientes fiéis sinalizados por engano (horizontal), em todos os limiares</li><li><b style='color:#3777FF'>Precision-Recall</b>: dos clientes sinalizados, quantos saíram de facto, à medida que se apanham mais churners. A linha pontilhada é a taxa de churn, o que se consegue ao acaso</li></ul></div></div>

In [ ]:
# oof já inclui o Churn Score IBM (benchmark) e a regra de negócio
model_plots.plot_roc_pr(y_train, oof, best=BEST, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.7 · Matrizes de confusão</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Para cada modelo, no limiar 0,5: nº de clientes e % da classe real. A diagonal são os acertos. O quadrado de <b style='color:#3777FF'>baixo à esquerda</b> são os churners que o modelo não detetou, o erro mais caro para a retenção; o de cima à direita são clientes fiéis contactados sem necessidade.</p></div></div>

In [ ]:
model_plots.plot_confusion_matrices(y_train, oof, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.8 · Limiar de decisão do modelo final</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>O modelo dá uma probabilidade; para agir é preciso um limiar. Baixar o limiar apanha mais churners (recall sobe) mas contacta mais clientes que ficariam (precision desce). A linha a tracejado marca o limiar que maximiza o F1, como referência.</p>
<p>Na prática, a decisão faz-se com a lista ordenada por <b style='color:#0EE071'>P(churn) × CLTV</b> e um orçamento de contactos (secção 10); o limiar serve para traduzir esse orçamento numa regra e para discutir custos com o Head of Customer Retention.</p></div></div>

In [ ]:
limiar_f1 = modelling.best_f1_threshold(y_train, oof[BEST])
limiares = modelling.threshold_table(y_train, oof[BEST])
limiares.to_csv(TAB_DIR / "11b_limiares_modelo_final.csv")
display(style_table(limiares, precision=3))
model_plots.plot_threshold(y_train, oof[BEST], BEST, marker=limiar_f1, show=SHOW)
print("Limiar que maximiza o F1 (out-of-fold):", round(limiar_f1, 2))

# Limiares de negócio: a equipa de retenção só consegue contactar os top 10% ou 20% da base.
# O limiar é o valor de P(churn) que sinaliza exatamente essa fração (fixado aqui, no OOF).
cortes = modelling.capacity_thresholds(oof[BEST], shares=(0.10, 0.20))
LIMIARES = {"0,5 (por omissão)": 0.5, "melhor F1": limiar_f1,
            "capacidade top 10%": cortes[0.10], "capacidade top 20%": cortes[0.20]}
limiares_negocio = modelling.decision_table(y_train, oof[BEST], LIMIARES)
limiares_negocio.to_csv(TAB_DIR / "11e_limiares_negocio_cv.csv")
display(style_table(limiares_negocio, precision=3))

# Prioridade de contacto (usada na app): top 10% / top 20% da base, pela capacidade da equipa.
# O nível de risco (baixo < 30%, médio 30–60%, alto ≥ 60%) é absoluto: modelling.RISK_LEVELS.
CAPACITY_CUTS = {"top 10%": cortes[0.10], "top 20%": cortes[0.20]}

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.9 · Como treinou a rede neuronal</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Treinamos a rede com os hiperparâmetros escolhidos e mostramos a evolução: a <b style='color:#3777FF'>log loss no treino</b> desce a cada época, e a <b style='color:#0EE071'>accuracy na validação interna</b> (15% do treino) sobe até estabilizar. A linha a tracejado é a melhor época; o treino pára 20 épocas depois, se não houver melhoria.</p></div></div>

In [ ]:
mlp_pipeline = models["Neural Network (MLP)"].fit(X_train, y_train)
mlp = mlp_pipeline.named_steps["clf"]
print("Arquitetura:", mlp.hidden_layer_sizes, "| ativação:", mlp.activation,
      "| épocas treinadas:", mlp.n_iter_)
model_plots.plot_mlp_training(mlp, show=SHOW)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>6 · Satisfaction Score — com e sem</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>No notebook 01, <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Satisfaction Score</code> 1–2 corresponde a 100% de churn: suspeita de ser registada no contexto da saída.
Treinamos o modelo selecionado <b style='color:#0EE071'>com e sem</b> a variável, no mesmo 5-fold, e documentamos a diferença.</p>
<p>Se o ganho for grande, é sinal de leakage: a versão com a variável <b style='color:#3777FF'>não serve para retenção preventiva</b>, porque o score só existe tarde demais.</p></div></div>

In [ ]:
# Modelo A: SEM Satisfaction Score (é o modelo escolhido na secção 5; o CV já foi feito)
cv_sem = cv_table.loc[[BEST], modelling.SUMMARY_COLS]   # AUC (± desvio), PR-AUC e Brier
cv_sem = cv_sem.rename(index={BEST: BEST + " — sem Satisfaction"})

# Modelo B: COM Satisfaction Score (igual, mas com mais uma coluna numérica)
NUM_SAT = NUM + ["Satisfaction Score"]
FEATURES_SAT = NUM_SAT + CAT
modelo_com = modelling.build_models(NUM_SAT, CAT, best_params)[BEST]
cv_com, _ = modelling.cv_evaluate({BEST + " — com Satisfaction": modelo_com}, train[FEATURES_SAT], y_train)
cv_com = cv_com[modelling.SUMMARY_COLS]

# Juntar as duas linhas numa só tabela
sat_cv = pd.concat([cv_sem, cv_com])
display(style_table(sat_cv, precision=4, highlight="auc"))

auc_sem = cv_sem["auc"].iloc[0]   # iloc[0] = primeira (e única) linha
auc_com = cv_com["auc"].iloc[0]
print("Diferença de AUC (com - sem):", round(auc_com - auc_sem, 4))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>7 · City com TargetEncoder</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>City</code> tem ~1.100 valores distintos: one-hot é inviável. O <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>TargetEncoder</code> substitui cada cidade pela taxa de churn suavizada,
calculada com <i>cross-fitting</i> e <b style='color:#0EE071'>dentro do Pipeline</b> — em cada fold só vê o treino desse fold.</p>
<p>Só entra no modelo final se melhorar o AUC em CV de forma clara (mais do que o desvio-padrão entre folds).</p></div></div>

In [ ]:
# Mesmo modelo, mas com a coluna City (codificada com TargetEncoder)
FEATURES_CITY = FEATURES + ["City"]
city_model = modelling.build_models(NUM, CAT, best_params, te=["City"])[BEST]
city_cv, _ = modelling.cv_evaluate({BEST + " + City (TargetEncoder)": city_model}, train[FEATURES_CITY], y_train)
city_cv = city_cv[modelling.SUMMARY_COLS]

variants = pd.concat([sat_cv, city_cv])
variants.to_csv(TAB_DIR / "12_variantes_satisfaction_city.csv")
display(style_table(variants, precision=4, highlight="auc"))

# Só usamos City se o ganho for maior do que o desvio-padrão do melhor modelo entre folds
gain, USE_CITY = modelling.include_city(cv_table, BEST, city_cv)

print("Ganho de AUC com City:", round(gain, 4))
print("Incluir City no modelo final?", USE_CITY)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>8 · Avaliação final no holdout</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>O holdout (20%) é usado <b style='color:#0EE071'>uma única vez</b> para avaliar o modelo core: todos os modelos são treinados no treino completo e avaliados no teste. As secções 10 e 11 voltam a usá-lo, já depois de todas as escolhas feitas, para avaliar o modelo de priorização e as decisões de retenção (lista completa em docs/modelling.md).
O modelo final já foi escolhido na secção 5; os restantes aparecem só para transparência.</p>
<p>O <b style='color:#3777FF'>Churn Score IBM</b> é avaliado no mesmo holdout — é a fasquia a bater. O seu Brier é dado apenas como referência: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Churn Score / 100</code> não é uma probabilidade calibrada. A versão com Satisfaction Score é reportada à parte, para documentar o efeito do leakage. A tabela tem as nove métricas para todos os modelos, incluindo a rede neuronal.</p></div></div>

In [ ]:
# Definir o modelo final e as suas colunas
if USE_CITY:
    final_model = city_model
    FINAL_FEATURES = FEATURES_CITY
else:
    final_model = models[BEST]
    FINAL_FEATURES = FEATURES

# Treinar cada modelo no treino completo e prever P(churn) no teste
probas = modelling.fit_predict(models, X_train, y_train, X_test)

# Se o modelo final usa City, treiná-lo com essas colunas e substituir a previsão
if USE_CITY:
    probas[BEST] = modelling.fit_predict({BEST: final_model}, train[FINAL_FEATURES], y_train,
                                         test[FINAL_FEATURES])[BEST]

# Versão com Satisfaction Score (só para mostrar o efeito do leakage)
probas[BEST + " + Satisfaction (leakage)"] = modelling.fit_predict(
    {"sat": modelo_com}, train[FEATURES_SAT], y_train, test[FEATURES_SAT])["sat"]

# Tabela de métricas no teste (inclui o Churn Score IBM)
holdout = modelling.holdout_report(probas, y_test, test["Churn Score"])
holdout.to_csv(TAB_DIR / "13_holdout.csv")
display(style_table(holdout, precision=4, highlight="auc"))

# Curva ROC e calibração
p_test = probas[BEST]   # P(churn) do modelo final para cada cliente do teste

# Os limiares de negócio fixados na CV (secção 5.8), aplicados uma vez ao holdout
limiares_holdout = modelling.decision_table(y_test, p_test, LIMIARES)
limiares_holdout.to_csv(TAB_DIR / "13b_limiares_negocio_holdout.csv")
display(style_table(limiares_holdout, precision=3))

# Reporte (não decide nada): IC 95% da diferença de AUC face ao IBM, por bootstrap emparelhado
# (os mesmos clientes reamostrados para os dois scores, 2000 reamostras, random_state fixo)
boot = modelling.paired_bootstrap_auc_diff(y_test, p_test, test["Churn Score"] / 100)
boot_ibm = pd.DataFrame([boot], index=[BEST + " − " + modelling.BENCHMARK])
boot_ibm.to_csv(TAB_DIR / "13d_bootstrap_auc_vs_ibm.csv")
display(style_table(boot_ibm, precision=4))
print("O intervalo inclui zero?", boot["ic_inclui_zero"])

# Níveis de risco absolutos (30% / 60%): como o modelo está calibrado, P ≈ churn observado
niveis_risco = modelling.risk_level_table(y_test, p_test)
niveis_risco.to_csv(TAB_DIR / "13c_niveis_risco_holdout.csv")
display(style_table(niveis_risco, precision=3, highlight="churn_real"))
curvas = {
    BEST: p_test,
    "Churn Score IBM": test["Churn Score"] / 100,
    "Logistic Regression": probas["Logistic Regression"],
    "Neural Network (MLP)": probas["Neural Network (MLP)"],
}
model_plots.plot_roc_calibration(y_test, curvas, show=SHOW)

# As nove métricas no holdout, modelo a modelo (só para transparência: a escolha já foi feita)
model_plots.plot_metrics_panel(holdout.drop(index=modelling.BASELINE), best=BEST,
                               name="24b_painel_metricas_holdout",
                               title="Nove métricas por modelo — holdout (20%)", show=SHOW)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>9 · Interpretação</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p><b style='color:#0EE071'>Permutation importance</b> no holdout: quanto cai o AUC quando os valores de uma variável são baralhados.
Funciona para qualquer modelo (incluindo ensembles) e ao nível das variáveis originais, não das colunas one-hot — mais útil para explicar ao Head of Customer Retention.</p>
<p>Barras <b style='color:#3777FF'>azuis</b> = numéricas, <b style='color:#0EE071'>verdes</b> = categóricas.</p></div></div>

In [ ]:
# Quanto cai o AUC quando baralhamos os valores de cada coluna?
# (5 repetições; valores negativos passam a 0; da mais para a menos importante)
importance = modelling.permutation_ranking(final_model, test[FINAL_FEATURES], y_test)
importance.to_frame("queda_auc").to_csv(TAB_DIR / "14_importancia_permutacao.csv")

titulo = "Permutation importance — " + BEST + " (queda de AUC, holdout)"
plots.plot_ranking(importance.head(20), titulo, "27_importancia_permutacao",
                   highlight=set(CAT + ["City"]), show=SHOW)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>10 · Modelo de priorização: clientes estabelecidos (tenure &gt; 3)</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>No notebook 01 (secção 6.1) vimos que, dentro de Churned + Stayed, <b style='color:#3777FF'>todos os clientes com tenure ≤ 3 meses saíram</b>. Não é comportamento: os clientes novos que ficaram estão rotulados <b style='color:#0EE071'>Joined</b> e foram excluídos da população core. O modelo core aprende por isso que "cliente recente = churn", e uma parte do seu AUC vem desses casos fáceis.</p>
<p>Mantemos as duas visões, cada uma com o seu papel:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>Modelo core</b> (secções 4 a 9): a população do enunciado, Churned + Stayed com todos os tenures. É o modelo que responde ao enunciado e que comparamos com o Churn Score IBM.</li><li><b style='color:#0EE071'>Modelo de priorização</b> (esta secção): o mesmo protocolo, mas treinado só com clientes <b style='color:#3777FF'>estabelecidos</b> (tenure &gt; 3). É o modelo que usamos para pontuar e ordenar os clientes estabelecidos na secção 11.</li></ul>
<p>O treino e o teste são os mesmos do modelo core, só filtrados por tenure: nenhum cliente do teste entra no treino. Os clientes novos não recebem P(churn) e são tratados por uma regra de onboarding (secção 12).</p></div></div>

In [ ]:
# O artefacto no treino: os clientes com tenure <= 3 saíram todos
novos_no_treino = ~features.is_established(train)
print("Clientes do treino com tenure <= 3:", novos_no_treino.sum(),
      "| taxa de churn:", round(train.loc[novos_no_treino, TARGET].mean(), 3))

# Os mesmos treino e teste do modelo core, só com clientes estabelecidos
train_est = train[features.is_established(train)]
test_est = test[features.is_established(test)]
X_train_est = train_est[FEATURES]
y_train_est = train_est[TARGET]
X_test_est = test_est[FEATURES]
y_test_est = test_est[TARGET]

print("Treino (tenure > 3):", len(train_est), "clientes | taxa de churn:", round(y_train_est.mean(), 3))
print("Teste (tenure > 3): ", len(test_est), "clientes | taxa de churn:", round(y_test_est.mean(), 3))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>10.1 · Tuning e seleção do modelo de priorização</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Repetimos as secções 4 e 5 nos clientes estabelecidos: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>RandomizedSearchCV</code> em 5-fold só no treino, os mesmos modelos e ensembles, e escolha pelo <b style='color:#0EE071'>maior ROC AUC em CV</b>. Os hiperparâmetros são afinados de novo porque a população mudou. O holdout continua sem ser usado.</p></div></div>

In [ ]:
best_params_est, tuning_est = modelling.tune_base_models(X_train_est, y_train_est, NUM, CAT,
                                                        n_iter=N_ITER)
tuning_est.to_csv(TAB_DIR / "16_tuning_estabelecidos.csv")

models_est = modelling.build_models(NUM, CAT, best_params_est)
ibm_est = {modelling.BENCHMARK: train_est["Churn Score"].to_numpy() / 100}
cv_est, oof_est = modelling.cv_evaluate(models_est, X_train_est, y_train_est, benchmarks=ibm_est)
cv_est.to_csv(TAB_DIR / "16b_cv_modelos_estabelecidos.csv")
display(style_table(modelling.metrics_view(cv_est), precision=3, highlight="auc"))

# O Churn Score IBM e a regra de negócio aparecem como referências, nos mesmos folds
model_plots.plot_cv_comparison(cv_est, name="28_cv_modelos_estabelecidos",
                               title="AUC em validação cruzada — clientes estabelecidos (tenure > 3)",
                               show=SHOW)

# A mesma regra de parcimónia do modelo core
BEST_EST, selecao_est, motivo_est = modelling.select_parsimonious(cv_est)
selecao_est.to_csv(TAB_DIR / "16c_selecao_modelo_estabelecidos.csv")
print("Modelo de priorização:", BEST_EST,
      "| AUC", round(cv_est.loc[BEST_EST, "auc"], 4), "±", round(cv_est.loc[BEST_EST, "auc_std"], 4))
print("Porquê:", motivo_est)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>10.2 · Modelo core vs modelo de priorização vs IBM</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Treinamos o modelo de priorização no treino completo (tenure &gt; 3) e avaliamos no holdout, <b style='color:#0EE071'>uma única vez</b>. A tabela compara os três scores em dois conjuntos de clientes do mesmo holdout:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>holdout completo</b>: a avaliação do enunciado, só para o modelo core e o IBM;</li><li><b style='color:#3777FF'>holdout tenure &gt; 3</b>: os clientes estabelecidos, onde os três podem ser comparados nas mesmas condições.</li></ul>
<p>Só faz sentido comparar linhas do mesmo conjunto. A diferença entre as duas linhas do modelo core mostra quanto do seu AUC vinha dos clientes novos. O Brier e o log loss do IBM são só indicativos: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Churn Score / 100</code> não é uma probabilidade calibrada.</p></div></div>

In [ ]:
priority_model = models_est[BEST_EST].fit(X_train_est, y_train_est)
p_test_est = priority_model.predict_proba(X_test_est)[:, 1]

# Capacidade da equipa medida só nos clientes estabelecidos (é este modelo que pontua o lote na
# app): os clientes novos, com P ≈ 1 no modelo core, já não ocupam o top 10% / 20%.
cortes_est = modelling.capacity_thresholds(oof_est[BEST_EST], shares=(0.10, 0.20))
LIMIARES_EST = {"0,5 (por omissão)": 0.5, "capacidade top 10%": cortes_est[0.10],
                "capacidade top 20%": cortes_est[0.20]}
CAPACITY_CUTS_EST = {"top 10%": cortes_est[0.10], "top 20%": cortes_est[0.20]}
modelling.decision_table(y_train_est, oof_est[BEST_EST], LIMIARES_EST).to_csv(
    TAB_DIR / "16d_limiares_capacidade_estabelecidos_cv.csv")
limiares_est = modelling.decision_table(y_test_est, p_test_est, LIMIARES_EST)
limiares_est.to_csv(TAB_DIR / "17b_limiares_capacidade_estabelecidos_holdout.csv")
display(style_table(limiares_est, precision=3))

# Reporte (não é gate): IC 95% da diferença de AUC face ao IBM nos clientes estabelecidos
boot_est = modelling.paired_bootstrap_auc_diff(y_test_est, p_test_est, test_est["Churn Score"] / 100)
boot_est_tab = pd.DataFrame([boot_est], index=[BEST_EST + " (priorização) − " + modelling.BENCHMARK])
boot_est_tab.to_csv(TAB_DIR / "17d_bootstrap_auc_priorizacao_vs_ibm.csv")
display(style_table(boot_est_tab, precision=4))

# Níveis de risco (30% / 60%) com o modelo de priorização: também calibrados
niveis_est = modelling.risk_level_table(y_test_est, p_test_est)
niveis_est.to_csv(TAB_DIR / "17c_niveis_risco_estabelecidos_holdout.csv")
display(style_table(niveis_est, precision=3, highlight="churn_real"))

# Para avaliar o modelo core nos mesmos clientes estabelecidos do holdout
est_mask = features.is_established(test).to_numpy()

comparison = modelling.comparison_table([
    ("Modelo core: " + BEST, "holdout completo", y_test, p_test),
    ("Churn Score IBM", "holdout completo", y_test, test["Churn Score"] / 100),
    ("Modelo core: " + BEST, "holdout tenure > 3", y_test[est_mask], p_test[est_mask]),
    ("Modelo de priorização: " + BEST_EST, "holdout tenure > 3", y_test_est, p_test_est),
    ("Churn Score IBM", "holdout tenure > 3", y_test_est, test_est["Churn Score"] / 100),
])
comparison.to_csv(TAB_DIR / "17_comparacao_core_priorizacao.csv")
display(style_table(comparison, precision=3, highlight="auc"))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>10.3 · Que variáveis pesam no modelo de priorização</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Permutation importance no holdout dos clientes estabelecidos, como na secção 9. A tabela compara as 10 variáveis mais importantes dos dois modelos: sem os clientes novos, <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Tenure in Months</code> deixa de carregar o artefacto e outras variáveis ganham peso.</p><p>A importância mede quanto o modelo <b style='color:#3777FF'>depende</b> de uma variável para ordenar os clientes. <b style='color:#0EE071'>Não mede o efeito causal</b> de mudar essa variável.</p></div></div>

In [ ]:
importance_est = modelling.permutation_ranking(priority_model, X_test_est, y_test_est)
importance_est.to_frame("queda_auc").to_csv(TAB_DIR / "18_importancia_priorizacao.csv")

titulo = "Permutation importance — " + BEST_EST + " (tenure > 3, holdout)"
plots.plot_ranking(importance_est.head(20), titulo, "29_importancia_priorizacao",
                   highlight=set(CAT), show=SHOW)

# As 10 mais importantes de cada modelo, lado a lado
top10 = pd.DataFrame({
    "modelo core": importance.head(10).index,
    "modelo de priorização": importance_est.head(10).index,
}, index=range(1, 11))
display(style_table(top10))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>11 · Priorização dos clientes estabelecidos: P(churn) × CLTV</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>O objetivo não é só prever churn, é decidir <b style='color:#3777FF'>quem contactar primeiro</b>. Cada cliente recebe uma
<b style='color:#0EE071'>perda esperada = P(churn) × CLTV</b> e a lista de intervenção é ordenada por esse valor.</p>
<p>Usamos o <b style='color:#0EE071'>modelo de priorização</b> (secção 10) nos clientes estabelecidos do holdout. A <b style='color:#0EE071'>curva de captura</b> mede, que % do CLTV dos clientes que de facto saíram fica coberta ao contactar os top-k% da lista.
Compara quatro ordenações: o nosso P × CLTV, só o nosso P(churn), o Churn Score IBM × CLTV e o Churn Score IBM sozinho.</p></div></div>

In [ ]:
# 4 formas diferentes de ordenar os clientes estabelecidos do teste
orderings = {
    "Modelo: P × CLTV": p_test_est * test_est["CLTV"],
    "Modelo: só P(churn)": p_test_est,
    "IBM: Churn Score × CLTV": test_est["Churn Score"] * test_est["CLTV"],
    "IBM: só Churn Score": test_est["Churn Score"],
}

# Que % do CLTV perdido apanhamos ao contactar os primeiros k% de cada lista?
curve = modelling.capture_curve(y_test_est, test_est["CLTV"], orderings)
capture = modelling.capture_at(curve, cuts=(0.1, 0.2, 0.3))
capture.to_csv(TAB_DIR / "15_captura_valor.csv")
display(style_table(capture, precision=3))
model_plots.plot_capture_curve(curve, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>11.1 · Só P(churn) ou P(churn) × CLTV?</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>As duas ordenações respondem a perguntas diferentes: <b style='color:#3777FF'>só P(churn)</b> contacta primeiro quem tem mais risco; <b style='color:#0EE071'>P × CLTV</b> contacta primeiro quem tem maior perda esperada. Para o mesmo nº de contactos comparamos:</p>
<ul style='margin:6px 0 2px 0'><li>quantos clientes as duas listas têm <b style='color:#3777FF'>em comum</b> (se for perto de 100%, a escolha quase não importa);</li><li>que % dos churners cada lista apanha;</li><li>que % do CLTV perdido cada lista cobre (o objetivo da retenção).</li></ul>
<p>A comparação com o IBM é feita da mesma forma: P × CLTV com Churn Score × CLTV, e só P com só Churn Score.</p></div></div>

In [ ]:
orderings_compared = retention.compare_orderings(
    y_test_est, test_est["CLTV"],
    score_p=p_test_est,
    score_value=p_test_est * test_est["CLTV"],
)
orderings_compared.to_csv(TAB_DIR / "15b_P_vs_PxCLTV.csv")
display(style_table(orderings_compared, precision=3, highlight="em_comum"))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>11.2 · Quantos clientes contactar? Análise custo-benefício</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>A lista diz <b style='color:#3777FF'>por que ordem</b> contactar. Para decidir <b style='color:#3777FF'>quantos</b>, é preciso comparar o valor salvo com o custo da campanha. Isso exige dois números que o dataset <b style='color:#0EE071'>não tem</b>:</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>taxa_sucesso</code>: % dos churners contactados que ficam por causa do contacto</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>custo_contacto</code>: custo de contactar um cliente, incluindo o incentivo, na mesma unidade do CLTV. Paga-se por todos os contactados, mesmo pelos que ficariam de qualquer forma</li></ul>
<p>São <b style='color:#0EE071'>pressupostos</b>, por isso ficam todos na célula de parâmetros abaixo, em três cenários. Os valores são ilustrativos e devem ser substituídos pelos da empresa.</p>
<p>Para cada cenário calculamos o nº de clientes a contactar de duas formas:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>Regra do valor esperado</b>: contactar quem tem P(churn) × CLTV × taxa_sucesso &gt; custo_contacto. Só usa o modelo, por isso é a regra que se pode aplicar a uma lista futura, sem conhecer o churn.</li><li><b style='color:#3777FF'>Ótimo observado</b>: o N que maximiza o valor líquido no holdout, usando o churn real. Serve para verificar a regra; não está disponível na prática.</li></ul>
<p>Se os dois N forem parecidos, as probabilidades do modelo estão suficientemente bem calibradas para decidir o tamanho da campanha.</p></div></div>

In [ ]:
# ==========================================================================
# PARÂMETROS DE NEGÓCIO — PRESSUPOSTOS, NÃO DADOS
# Definidos uma só vez em src/telco_churn/retention.py (CENARIOS), para o notebook e o
# script usarem os mesmos valores. Substituir lá pelos valores reais da empresa.
#   taxa_sucesso   : % dos churners contactados que ficam por causa do contacto
#   custo_contacto : custo por cliente contactado (contacto + incentivo), na unidade do CLTV
# ==========================================================================
CENARIOS = retention.CENARIOS
display(style_table(pd.DataFrame(CENARIOS).T, precision=2))

In [ ]:
scenarios = retention.scenario_table(y_test_est, p_test_est, test_est["CLTV"], CENARIOS)
scenarios.to_csv(TAB_DIR / "20_cenarios_custo_beneficio.csv")
display(style_table(scenarios, precision=2))

# Curva de valor líquido de cada cenário (lista ordenada por P × CLTV)
value_curves = {}
for name, params in CENARIOS.items():
    value_curves[name] = retention.net_value_curve(
        y_test_est, test_est["CLTV"], p_test_est * test_est["CLTV"],
        params["taxa_sucesso"], params["custo_contacto"])
model_plots.plot_net_value(value_curves, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>11.2.1 · Sensibilidade aos pressupostos</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Os três cenários são só três pontos. A grelha mostra a <b style='color:#3777FF'>% da base a contactar</b> pela regra do valor esperado para várias combinações de taxa de sucesso e custo. Ajuda a perceber quais os pressupostos que mais mudam a decisão e onde vale a pena medir melhor (por exemplo, com um piloto).</p></div></div>

In [ ]:
sensitivity = retention.sensitivity_grid(
    p_test_est, test_est["CLTV"],
    success_rates=[0.05, 0.10, 0.20, 0.30, 0.40],
    costs=[25, 50, 100, 150, 200],
)
sensitivity.to_csv(TAB_DIR / "20b_sensibilidade_pct_contactar.csv")
display(style_table(sensitivity, precision=2))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>11.3 · Lista de intervenção</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Lista ordenada dos clientes estabelecidos do holdout, onde podemos verificar o resultado. O mapa mostra onde ficam os top-N: o P × CLTV troca alguns clientes de risco alto mas pouco valor por clientes de valor alto com risco moderado.</p>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>TOP_N</code> é o orçamento de contactos — ajustar ao que a equipa de retenção consegue tratar.</p>
<p>Os <b style='color:#3777FF'>Joined</b> já não recebem P(churn): seria extrapolar um modelo que aprendeu o artefacto do tenure. São tratados na secção 12.</p></div></div>

In [ ]:
TOP_N = 200   # nº de clientes que a equipa de retenção consegue contactar

# Lista de prioridade dos clientes estabelecidos do teste + churn real (para verificar)
prio_test = modelling.priority_list(test_est, p_test_est)
churn_real = test_est[["Customer ID", TARGET]]
prio_test = prio_test.merge(churn_real, on="Customer ID")
model_plots.plot_priority_map(prio_test, TOP_N, show=SHOW)

# Resumo por decil (decil 1 = 10% de clientes com maior perda esperada)
grupos = prio_test.groupby("decil")
by_decile = pd.DataFrame({
    "clientes": grupos.size(),
    "p_churn_media": grupos["p_churn"].mean(),
    "churn_real": grupos[TARGET].mean(),
    "cltv_medio": grupos["CLTV"].mean(),
    "perda_esperada": grupos["perda_esperada"].sum(),
})
display(style_table(by_decile, precision=3, highlight="churn_real"))
display(style_table(prio_test.head(15), precision=3))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>11.4 · Que ação propor? Hipóteses por segmento</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Cada cliente da lista recebe uma <b style='color:#3777FF'>ação sugerida</b>, com regras simples sobre variáveis que a empresa conhece antes do churn (<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>retention.suggest_action</code>). As regras inspiram-se na EDA e na importância das variáveis:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>Proposta de contrato anual</b>: clientes Month-to-Month, onde se concentra o churn</li><li><b style='color:#3777FF'>Oferta de suporte técnico e segurança</b>: clientes com fibra sem Online Security nem Premium Tech Support</li><li><b style='color:#3777FF'>Revisão de tarifário antes da renovação</b>: restantes, que já têm contrato de 1 ou 2 anos</li></ul>
<p><b style='color:#0EE071'>Atenção: são hipóteses a testar, não efeitos demonstrados.</b> A importância das variáveis mostra que o modelo <i>usa</i> o tipo de contrato para ordenar clientes; não prova que mudar o contrato de um cliente o faça ficar. O churn real por segmento (abaixo) é descritivo.</p>
<p><b style='color:#3777FF'>Nota sobre o código:</b> as regras usam <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>np.select(condições, valores, default)</code>, que testa as condições por ordem e atribui a cada cliente o valor da primeira condição verdadeira; quem não cumpre nenhuma recebe o default. É o equivalente a uma sequência de if / elif / else aplicada a todas as linhas.</p>
<p>Para medir o efeito de cada ação: em cada segmento, sortear uma parte dos clientes (por exemplo 20%) como <b style='color:#3777FF'>grupo de controlo</b>, que não é contactado. Ao fim de um trimestre, a diferença de churn entre contactados e controlo estima o efeito da ação, e é esse valor que deve substituir a <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>taxa_sucesso</code> assumida na secção 11.2.</p></div></div>

In [ ]:
# Juntar à lista as variáveis usadas nas regras e atribuir a ação sugerida
prio_test = prio_test.merge(test_est[["Customer ID"] + retention.ACTION_COLS], on="Customer ID")
prio_test["acao_sugerida"] = retention.suggest_action(prio_test)

# Orçamento: o N da regra do valor esperado no cenário Base (secção 11.2)
N_CAMPANHA = int(scenarios.loc["Base", "N_regra_valor_esperado"])
actions = retention.actions_summary(prio_test, N_CAMPANHA, TARGET)
actions.to_csv(TAB_DIR / "21_acoes_sugeridas.csv")
print("Top", N_CAMPANHA, "da lista (cenário Base), por ação sugerida:")
display(style_table(actions, precision=2, highlight="clientes"))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>12 · Clientes novos: regra de onboarding</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Os <b style='color:#3777FF'>Joined</b> (454 clientes, 1 a 3 meses de casa) não recebem P(churn). O modelo core daria a quase todos P ≈ 1, porque na população de treino todos os clientes novos saíram (secção 10), e o modelo de priorização nunca viu clientes novos.</p>
<p>Em vez disso, propomos uma <b style='color:#0EE071'>regra de onboarding</b> tirada da EDA. Comparamos os clientes novos que saíram (Churned com tenure ≤ 3) com os que ficaram (Joined) e procuramos as variáveis onde a taxa de saída mais muda.</p>
<p><b style='color:#0EE071'>É uma hipótese de negócio, não um resultado validado.</b> A taxa de saída é descritiva: depende de como o snapshot rotula os clientes e não é uma probabilidade. Para a validar, a regra deve ser aplicada a uma parte dos novos clientes, mantendo outra parte como grupo de controlo, e comparar a saída nos dois grupos ao fim de um trimestre.</p></div></div>

In [ ]:
# Base completa, com Customer Status, para juntar os novos que saíram e os que ficaram
merged = pd.read_csv(DATA_PROCESSED / "telco_merged_full.csv", keep_default_na=False, na_values=[""])
novos = merged[~features.is_established(merged)].copy()
novos["saiu"] = (novos["Customer Status"] == "Churned").astype(int)
print("Clientes novos (tenure <= 3):", len(novos))
display(style_table(novos["Customer Status"].value_counts().to_frame("clientes")))

# Taxa de saída dos clientes novos por contrato e por tipo de internet
for col in ["Contract", "Internet Type"]:
    by_col = novos.groupby(col)["saiu"].agg(clientes="size", taxa_saida="mean")
    display(style_table(by_col, precision=2, highlight="taxa_saida"))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>12.1 · A regra e a lista de onboarding</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>A saída dos clientes novos concentra-se nos contratos <b style='color:#3777FF'>Month-to-Month</b> e, dentro destes, na <b style='color:#3777FF'>fibra ótica</b>. Daqui saem três segmentos, definidos em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>retention.onboarding_segment</code>:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#0EE071'>Onboarding prioritário</b>: Month-to-Month com Fiber Optic — por exemplo, contacto de boas-vindas e verificação da qualidade do serviço no 1.º mês</li><li><b style='color:#3777FF'>Onboarding standard</b>: restantes Month-to-Month — por exemplo, comunicação automática com as vantagens de um contrato anual</li><li>Sem ação específica: contratos de 1 ou 2 anos, onde a saída de clientes novos é rara</li></ul>
<p>As ações são exemplos para discutir com o Head of Customer Retention. Dentro de cada segmento, a lista ordena por CLTV.</p></div></div>

In [ ]:
# Segmento de cada cliente novo, taxa de saída por segmento e nº de Joined em cada um
novos["onboarding"] = retention.onboarding_segment(novos)
regra = retention.onboarding_rule(novos)
regra.to_csv(TAB_DIR / "19_regra_onboarding.csv")
display(style_table(regra, precision=2, highlight="taxa_saida"))

# Lista de onboarding: os Joined, por segmento e CLTV
onboarding_joined = retention.onboarding_list(joined)
display(style_table(onboarding_joined.head(15), precision=0))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>13 · Exportação</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Ficheiros gerados (ignorados pelo Git):</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/artifacts/final_model.joblib</code> — modelo final treinado, lista de features e métricas</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/artifacts/priority_model.joblib</code> — modelo de priorização (tenure &gt; 3)</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/prioridade_holdout.csv</code> — lista de intervenção dos clientes estabelecidos do holdout</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/onboarding_joined.csv</code> — lista de onboarding dos Joined, por segmento</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/tables/10_*</code> a <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>19_*</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/figures/20_*</code> a <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>29_*</code> — tabelas e gráficos desta fase (lista completa em docs/modelling.md)</li></ul></div></div>

In [ ]:
# Guardar o modelo final e a informação necessária para o voltar a usar
modelling.save_bundle(ARTIFACTS_DIR / "final_model.joblib", final_model, BEST, FINAL_FEATURES,
                      TARGET, best_params, cv=cv_table.loc[BEST].to_dict(),
                      holdout=holdout.loc[BEST].to_dict(), selection=motivo,
                      benchmark_cv=cv_table.loc[modelling.BENCHMARK].to_dict(),
                      benchmark_holdout=holdout.loc[modelling.BENCHMARK].to_dict(),
                      form=scoring.form_spec(train, [c for c in FINAL_FEATURES
                                                     if c not in features.NEW_NUM + features.NEW_CAT]),
                      cltv_median=float(train["CLTV"].median()),
                      thresholds=LIMIARES, risk_levels=modelling.RISK_LEVELS,
                      capacity_cuts=CAPACITY_CUTS)

# Modelo de priorização (clientes estabelecidos, tenure > 3)
modelling.save_bundle(
    ARTIFACTS_DIR / "priority_model.joblib", priority_model, BEST_EST, FEATURES, TARGET,
    best_params_est, cv=cv_est.loc[BEST_EST].to_dict(), selection=motivo_est,
    holdout=comparison.loc[("holdout tenure > 3", "Modelo de priorização: " + BEST_EST)].to_dict(),
    benchmark_cv=cv_est.loc[modelling.BENCHMARK].to_dict(),
    benchmark_holdout=comparison.loc[("holdout tenure > 3", "Churn Score IBM")].to_dict(),
    bootstrap_vs_ibm=boot_est,
    thresholds=LIMIARES_EST, risk_levels=modelling.RISK_LEVELS, capacity_cuts=CAPACITY_CUTS_EST,
    form=scoring.form_spec(train_est, [c for c in FEATURES
                                       if c not in features.NEW_NUM + features.NEW_CAT]),
    cltv_median=float(train_est["CLTV"].median()),
    population="Churned + Stayed com tenure > 3")

# Guardar a lista de prioridade (estabelecidos) e a de onboarding (Joined)
prio_test.to_csv(DATA_PROCESSED / "prioridade_holdout.csv", index=False)
onboarding_joined.to_csv(DATA_PROCESSED / "onboarding_joined.csv", index=False)

print("Guardado: outputs/artifacts/final_model.joblib")
print("Guardado: outputs/artifacts/priority_model.joblib")
print("Guardado: data/processed/prioridade_holdout.csv")
print("Guardado: data/processed/onboarding_joined.csv")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>14 · Conclusões para o relatório</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>A célula seguinte resume os números-chave a partir das tabelas acima, para transcrever para o relatório:</p>
<ul style='margin:6px 0 2px 0'><li>modelos empatados com o vencedor em CV e desempenho da rede neuronal</li><li>desempenho do modelo final vs <b style='color:#3777FF'>Churn Score IBM</b> no mesmo holdout</li><li>efeito do <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Satisfaction Score</code> (leakage) e decisão sobre <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>City</code></li><li>ganho de valor captado com a priorização <b style='color:#0EE071'>P(churn) × CLTV</b></li></ul></div></div>

In [ ]:
# Números-chave para o relatório
auc_modelo = holdout.loc[BEST, "auc"]
auc_ibm = holdout.loc[modelling.BENCHMARK, "auc"]
pr_auc = holdout.loc[BEST, "pr_auc"]
brier = holdout.loc[BEST, "brier"]
auc_sat = holdout.loc[BEST + " + Satisfaction (leakage)", "auc"]

# Comparar sempre ordenações do mesmo tipo: P × CLTV com P × CLTV, só P com só P
captura_modelo = capture.loc["Modelo: P × CLTV", "top 20%"]
captura_ibm = capture.loc["IBM: Churn Score × CLTV", "top 20%"]
captura_modelo_p = capture.loc["Modelo: só P(churn)", "top 20%"]
captura_ibm_p = capture.loc["IBM: só Churn Score", "top 20%"]

print("Modelo final:", BEST)
print("  Seleção:", motivo)
print("  Regra de negócio (sem ML): AUC teste", round(holdout.loc[modelling.RULE_BASELINE, "auc"], 3))
print("  Rede neuronal (MLP): AUC CV", round(cv_table.loc["Neural Network (MLP)", "auc"], 3),
      "| AUC teste", round(holdout.loc["Neural Network (MLP)", "auc"], 3))
print("  AUC no teste:", round(auc_modelo, 3))
print("  AUC do IBM:  ", round(auc_ibm, 3))
print("  Diferença:   ", round(auc_modelo - auc_ibm, 3))
print("  PR-AUC:", round(pr_auc, 3), "| Brier:", round(brier, 3))
print("Modelo de priorização (tenure > 3):", BEST_EST)
auc_est = comparison.loc[("holdout tenure > 3", "Modelo de priorização: " + BEST_EST), "auc"]
auc_core_est = comparison.loc[("holdout tenure > 3", "Modelo core: " + BEST), "auc"]
auc_ibm_est = comparison.loc[("holdout tenure > 3", "Churn Score IBM"), "auc"]
print("  AUC no teste (tenure > 3): priorização", round(auc_est, 3), "| core", round(auc_core_est, 3),
      "| IBM", round(auc_ibm_est, 3))
print("Satisfaction Score: AUC com a variável =", round(auc_sat, 3), "(fora do modelo final)")
print("City incluída no modelo final?", USE_CITY, "(ganho CV:", round(gain, 4), ")")
print("Contactando 20% dos clientes estabelecidos (% do CLTV perdido coberto):")
print("  P × CLTV: modelo", round(captura_modelo * 100, 1), "% | IBM", round(captura_ibm * 100, 1), "%")
print("  Só P:     modelo", round(captura_modelo_p * 100, 1), "% | IBM", round(captura_ibm_p * 100, 1), "%")
print("  Clientes em comum entre a lista por P e por P × CLTV:",
      round(orderings_compared.loc["top 20%", "em_comum"] * 100, 1), "%")
print("Custo-benefício (pressupostos ilustrativos, secção 11.2):")
for name in scenarios.index:
    print("  ", name, "| contactar", scenarios.loc[name, "N_regra_valor_esperado"],
          "clientes do holdout (", round(scenarios.loc[name, "pct_base_regra"] * 100, 1), "% ) | ótimo observado:",
          scenarios.loc[name, "N_otimo_observado"])